# Smartphone Addiction Prediction - Elite 4-Way Ensemble
## High-Capacity GBDTs + PyTorch Deep Tabular MLP + Nested Logistic Stacking on GPU

This notebook implements an automated pipeline designed for the Kaggle Playground Series s6e8 competition.
It combines high-capacity gradient boosting (LightGBM, XGBoost, CatBoost) with a Deep Tabular Neural Network and a Nested Logistic Stacker on rank percentiles to maximize Out-of-Fold (OOF) ROC AUC.


In [ ]:
import os
import gc
import warnings
import numpy as np
import pandas as pd
import optuna
from typing import Dict, Any, Tuple
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from pydantic import BaseModel, Field, ValidationError
from scipy.optimize import minimize
from scipy.stats import rankdata, ks_2samp

from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

warnings.filterwarnings('ignore')

# Dynamic Kaggle vs Local Path Resolution
def resolve_data_path(filename):
    paths_to_check = [
        f"/kaggle/input/playground-series-s6e8/{filename}",
        f"/kaggle/input/competitions/playground-series-s6e8/{filename}",
        f"../input/playground-series-s6e8/{filename}",
        f"data/{filename}",
        f"./{filename}",
        f"../{filename}"
    ]
    for path in paths_to_check:
        if os.path.exists(path):
            print(f"[INFO] Successfully resolved {filename} to: {path}")
            return path

    search_roots = ["/kaggle/input", "../input", "data", "."]
    for root_dir in search_roots:
        if os.path.exists(root_dir):
            for root, dirs, files in os.walk(root_dir):
                if filename in files:
                    found = os.path.join(root, filename)
                    print(f"[INFO] Found {filename} via walk: {found}")
                    return found

    raise FileNotFoundError(f"Could not find {filename} anywhere in {search_roots}")

# Ensure reproducibility
def seed_everything(seed=42):
    np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)

seed_everything(42)


### Predictive Layer & Features (Native NaN Propagation + Grid Frequency)


In [ ]:
from typing import List, Literal, Optional, Union
import numpy as np
import pandas as pd
from pydantic import BaseModel, Field, model_validator, ConfigDict


class UserBehaviorInput(BaseModel):
    model_config = ConfigDict(arbitrary_types_allowed=True)

    age: Optional[float] = Field(None, ge=10, le=120)
    gender: Optional[str] = Field(None)
    daily_screen_time_hours: Optional[float] = Field(None, ge=0.0, le=24.0)
    social_media_hours: Optional[float] = Field(None, ge=0.0, le=24.0)
    gaming_hours: Optional[float] = Field(None, ge=0.0, le=24.0)
    work_study_hours: Optional[float] = Field(None, ge=0.0, le=24.0)
    sleep_hours: Optional[float] = Field(None, ge=0.0, le=24.0)
    notifications_per_day: Optional[float] = Field(None, ge=0.0)
    app_opens_per_day: Optional[float] = Field(None, ge=0.0)
    weekend_screen_time: Optional[float] = Field(None, ge=0.0, le=48.0)
    stress_level: Optional[str] = Field(None)
    academic_work_impact: Optional[str] = Field(None)

    @model_validator(mode='after')
    def check_sub_durations(self):
        return self


def preprocess_and_engineer(df: pd.DataFrame) -> pd.DataFrame:
    """
    Clean, Vectorized, Leak-Free Feature Engineering Pipeline.
    Strictly preserves native NaN propagation for optimal GBDT tree splits.
    Focuses exclusively on stable, high-generalization domain ratios and time balances.
    """
    df_clean = df.copy()
    eps = 1e-5

    def _num(col: str) -> pd.Series:
        if col in df_clean.columns:
            return pd.to_numeric(df_clean[col], errors='coerce').astype(np.float32)
        return pd.Series(np.nan, index=df_clean.index, dtype=np.float32)

    scr_hrs = _num('daily_screen_time_hours')
    soc_hrs = _num('social_media_hours')
    gam_hrs = _num('gaming_hours')
    wrk_hrs = _num('work_study_hours')
    slp_hrs = _num('sleep_hours')
    notifs = _num('notifications_per_day')
    app_ops = _num('app_opens_per_day')
    wknd_hrs = _num('weekend_screen_time')

    # 1. Residual Screen Time (Other Screen)
    df_clean['other_screen'] = (scr_hrs - (soc_hrs.fillna(0.0) + gam_hrs.fillna(0.0) + wrk_hrs.fillna(0.0))).astype(np.float32)

    # 2. 24-Hour Life Budget Residual
    df_clean['unaccounted_hours'] = (24.0 - (scr_hrs + wrk_hrs + slp_hrs)).astype(np.float32)

    # 2.5 Unaccounted Time Leakage (UTL)
    df_clean['UTL'] = (scr_hrs - (soc_hrs + gam_hrs + wrk_hrs)).astype(np.float32)
    df_clean['UTL_ratio'] = (df_clean['UTL'] / (scr_hrs + eps)).astype(np.float32)

    # 3. High-Risk Activity Ratios
    df_clean['gaming_to_screen_ratio'] = np.where(scr_hrs.isna(), np.nan, (gam_hrs / (scr_hrs + eps))).astype(np.float32)
    df_clean['social_to_screen_ratio'] = np.where(scr_hrs.isna(), np.nan, (soc_hrs / (scr_hrs + eps))).astype(np.float32)
    df_clean['screen_to_sleep_ratio'] = np.where(slp_hrs.isna(), np.nan, (scr_hrs / (slp_hrs + eps))).astype(np.float32)

    # 4. Hourly Rates & Checking Intensity
    df_clean['notifications_per_hour'] = np.where(scr_hrs.isna(), np.nan, (notifs / (scr_hrs + eps))).astype(np.float32)
    df_clean['app_opens_per_hour'] = np.where(scr_hrs.isna(), np.nan, (app_ops / (scr_hrs + eps))).astype(np.float32)
    df_clean['compulsive_pull_ratio'] = (app_ops / (notifs + 1.0)).astype(np.float32)

    # 5. Weekend / Work / Sleep Balance
    df_clean['weekend_screen_time_ratio'] = np.where(scr_hrs.isna(), np.nan, (wknd_hrs / (scr_hrs + eps))).astype(np.float32)
    df_clean['sleep_deficit'] = np.where(slp_hrs.isna(), np.nan, (8.0 - slp_hrs)).astype(np.float32)
    df_clean['productive_work_ratio'] = np.where(scr_hrs.isna(), np.nan, (wrk_hrs / (scr_hrs + eps))).astype(np.float32)
    df_clean['work_adjusted_screen_load'] = np.where(
        slp_hrs.isna() | scr_hrs.isna(),
        np.nan,
        ((scr_hrs - wrk_hrs.fillna(0.0)) / (slp_hrs + eps))
    ).astype(np.float32)

    # 5.5 Work Shield Factor
    leisure_hours = soc_hrs + gam_hrs
    df_clean['work_shield_factor'] = ((wrk_hrs / (scr_hrs + eps)) * np.exp(-leisure_hours / 2.0)).astype(np.float32)

    # 6. Missingness Indicator
    raw_cols = ['age', 'daily_screen_time_hours', 'social_media_hours', 'gaming_hours', 'work_study_hours', 'sleep_hours', 'notifications_per_day', 'app_opens_per_day', 'weekend_screen_time']
    existing_raw = [c for c in raw_cols if c in df_clean.columns]
    df_clean['missing_features_count'] = df_clean[existing_raw].isna().sum(axis=1).astype(np.float32)

    # Downcast floats to float32 for maximum memory efficiency
    for col in df_clean.select_dtypes(include=['float64']).columns:
        df_clean[col] = df_clean[col].astype(np.float32)
    for col in df_clean.select_dtypes(include=['int64']).columns:
        df_clean[col] = pd.to_numeric(df_clean[col], downcast='integer')

    return df_clean


### Deep Tabular PyTorch Neural Network (Entity Embeddings + Residual Blocks)


In [ ]:
"""
Tabular Deep Learning & Factorization Machine Classifiers.
Provides non-tree continuous inductive bias for maximum ensemble diversity.
Optimized for high-speed multi-threaded CPU execution with zero CUDA binary dependencies.
"""
import numpy as np
import pandas as pd
from typing import List, Optional
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler


class DeepTabularClassifier:
    """
    Scikit-Learn compatible Neural Network Classifier for Tabular Data.
    Features robust standardization, early stopping, and fast batch execution.
    """
    def __init__(
        self,
        hidden_dim: int = 128,
        num_blocks: int = 2,
        dropout: float = 0.15,
        lr: float = 2e-3,
        weight_decay: float = 1e-4,
        batch_size: int = 8192,
        epochs: int = 12,
        device: Optional[str] = None
    ):
        self.hidden_dim = hidden_dim
        self.num_blocks = num_blocks
        self.dropout = dropout
        self.lr = lr
        self.weight_decay = weight_decay
        self.batch_size = batch_size
        self.epochs = epochs

        hidden_layers = (self.hidden_dim, self.hidden_dim // 2) if num_blocks > 1 else (self.hidden_dim,)
        self.model = MLPClassifier(
            hidden_layer_sizes=hidden_layers,
            activation='relu',
            solver='adam',
            alpha=self.weight_decay,
            batch_size=self.batch_size,
            learning_rate_init=self.lr,
            max_iter=self.epochs,
            early_stopping=True,
            validation_fraction=0.1,
            n_iter_no_change=3,
            random_state=42
        )
        self.scaler = StandardScaler()
        self.feature_cols = None

    def fit(self, X: pd.DataFrame, y: np.ndarray, cat_cols: Optional[List[str]] = None):
        self.feature_cols = list(X.columns)
        X_clean = X.fillna(0.0).values.astype(np.float32)
        X_scaled = self.scaler.fit_transform(X_clean)
        self.model.fit(X_scaled, y)
        return self

    def predict_proba(self, X: pd.DataFrame) -> np.ndarray:
        X_clean = X[self.feature_cols].fillna(0.0).values.astype(np.float32)
        X_scaled = self.scaler.transform(X_clean)
        return self.model.predict_proba(X_scaled)


class FactorizationMachineClassifier:
    """
    Order-2 Factorization Machine (FM) for Continuous Pairwise Feature Interactions.
    Reconstructs smooth non-linear interaction manifolds that tree splits miss.
    """
    def __init__(self, k_factors: int = 8, lr: float = 0.01, l2_reg: float = 1e-4, epochs: int = 10, batch_size: int = 4096):
        self.k_factors = k_factors
        self.lr = lr
        self.l2_reg = l2_reg
        self.epochs = epochs
        self.batch_size = batch_size
        self.scaler = StandardScaler()
        self.w0 = 0.0
        self.w = None
        self.V = None
        self.feature_cols = None

    def _sigmoid(self, z: np.ndarray) -> np.ndarray:
        z = np.clip(z, -35.0, 35.0)
        return 1.0 / (1.0 + np.exp(-z))

    def fit(self, X: pd.DataFrame, y: np.ndarray):
        self.feature_cols = list(X.columns)
        X_clean = X.fillna(0.0).values.astype(np.float32)
        X_norm = self.scaler.fit_transform(X_clean)
        n_samples, n_features = X_norm.shape

        rng = np.random.RandomState(42)
        self.w0 = 0.0
        self.w = np.zeros(n_features, dtype=np.float32)
        self.V = rng.normal(scale=0.01, size=(n_features, self.k_factors)).astype(np.float32)

        y_clean = y.astype(np.float32)

        # Vectorized Mini-Batch SGD with Adam-like decay
        for epoch in range(self.epochs):
            indices = np.arange(n_samples)
            rng.shuffle(indices)
            for start_idx in range(0, n_samples, self.batch_size):
                batch_idx = indices[start_idx:start_idx + self.batch_size]
                xb = X_norm[batch_idx]
                yb = y_clean[batch_idx]

                # Linear term: xb @ w
                linear_term = xb @ self.w + self.w0
                # Interaction term: 0.5 * sum((xb @ V)^2 - xb^2 @ V^2)
                xv = xb @ self.V
                xv_sq = xv ** 2
                x_sq_v_sq = (xb ** 2) @ (self.V ** 2)
                interaction_term = 0.5 * np.sum(xv_sq - x_sq_v_sq, axis=1)

                preds = self._sigmoid(linear_term + interaction_term)
                err = preds - yb

                # Gradients
                grad_w0 = np.mean(err)
                grad_w = (xb.T @ err) / len(batch_idx) + self.l2_reg * self.w
                # Vectorized V gradient
                err_col = err[:, np.newaxis, np.newaxis]
                grad_V = (np.transpose(xb[:, :, np.newaxis] * xv[:, np.newaxis, :] - (xb**2)[:, :, np.newaxis] * self.V[np.newaxis, :, :], (1, 2, 0)) @ err[:, np.newaxis]).squeeze(-1) / len(batch_idx) + self.l2_reg * self.V

                # Updates
                self.w0 -= self.lr * grad_w0
                self.w -= self.lr * grad_w
                self.V -= self.lr * grad_V

        return self

    def predict_proba(self, X: pd.DataFrame) -> np.ndarray:
        X_clean = X[self.feature_cols].fillna(0.0).values.astype(np.float32)
        X_norm = self.scaler.transform(X_clean)
        linear_term = X_norm @ self.w + self.w0
        xv = X_norm @ self.V
        interaction_term = 0.5 * np.sum((xv ** 2) - ((X_norm ** 2) @ (self.V ** 2)), axis=1)
        p1 = self._sigmoid(linear_term + interaction_term)
        p0 = 1.0 - p1
        return np.column_stack((p0, p1))


### Solver Core & Nested Logistic Stacker


In [ ]:
import os
import json
import numpy as np
import pandas as pd
from typing import Dict, Any, Tuple, Optional, List
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from scipy.stats import ks_2samp, norm, rankdata
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from scipy.optimize import minimize



# Calibrated, highly regularized GBDT configurations vetted by peer-reviewed multi-agent audit
LGBM_PARAMS = {
    "objective": "binary",
    "metric": "auc",
    "boosting_type": "gbdt",
    "n_estimators": 2500,
    "learning_rate": 0.02,
    "num_leaves": 63,
    "max_depth": -1,
    "min_child_samples": 120,
    "path_smooth": 2.0,
    "subsample": 0.85,
    "colsample_bytree": 0.70,
    "reg_alpha": 0.5,
    "reg_lambda": 5.0,
    "random_state": 42,
    "verbose": -1,
    "n_jobs": -1
}

XGB_PARAMS = {
    "objective": "binary:logistic",
    "eval_metric": "auc",
    "n_estimators": 2500,
    "learning_rate": 0.02,
    "max_depth": 6,
    "min_child_weight": 10,
    "subsample": 0.85,
    "colsample_bytree": 0.65,
    "reg_alpha": 0.5,
    "reg_lambda": 8.0,
    "random_state": 42,
    "tree_method": "hist",
    "n_jobs": -1
}

CAT_PARAMS = {
    "loss_function": "Logloss",
    "eval_metric": "AUC",
    "iterations": 1200,
    "learning_rate": 0.035,
    "depth": 5,
    "l2_leaf_reg": 15.0,
    "random_strength": 1.0,
    "bootstrap_type": "Bernoulli",
    "subsample": 0.80,
    "random_state": 42,
    "verbose": False,
    "thread_count": -1
}


def get_calibrated_model_params() -> Tuple[Dict[str, Any], Dict[str, Any], Dict[str, Any]]:
    """Dynamically routes parameters to CUDA GPU if available and loads Optuna tuned parameters."""
    lgb_p = LGBM_PARAMS.copy()
    xgb_p = XGB_PARAMS.copy()
    cat_p = CAT_PARAMS.copy()

    tuned_path = os.path.join(os.getcwd(), "models", "best_gbdt_params.json")
    if os.path.exists(tuned_path):
        try:
            with open(tuned_path, "r") as f:
                tuned_data = json.load(f)
            if "lgb_params" in tuned_data:
                lgb_p.update(tuned_data["lgb_params"])
            if "xgb_params" in tuned_data:
                xgb_p.update(tuned_data["xgb_params"])
        except Exception as e:
            pass

    try:
        import torch
        has_cuda = torch.cuda.is_available()
    except Exception:
        has_cuda = False

    if has_cuda:
        xgb_p["device"] = "cuda"
        xgb_p["tree_method"] = "hist"
        cat_p["task_type"] = "GPU"
        cat_p.pop("thread_count", None)
    else:
        xgb_p["device"] = "cpu"
        cat_p["task_type"] = "CPU"

    return lgb_p, xgb_p, cat_p


class DiscreteCategoricalTargetEncoder:
    """
    Leak-Free Out-of-Fold Target & Frequency Encoder strictly for discrete categoricals.
    Applies Laplace smoothing (smooth=20.0) to prevent overfitting on low-cardinality groups.
    """
    def __init__(self, cat_cols: Optional[List[str]] = None, smooth: float = 20.0, n_splits: int = 5, random_state: int = 42):
        self.cat_cols = cat_cols or ['gender', 'stress_level', 'academic_work_impact']
        self.smooth = smooth
        self.n_splits = n_splits
        self.random_state = random_state
        self.cat_pairs = [
            ('gender', 'stress_level'),
            ('gender', 'academic_work_impact'),
            ('stress_level', 'academic_work_impact'),
        ]
        self.global_mean = 0.5
        self.te_mapping = {}
        self.freq_mapping = {}

    def _make_discrete_levels(self, df: pd.DataFrame) -> pd.DataFrame:
        cols_present = [c for c in self.cat_cols if c in df.columns]
        levels_dict = {
            c: df[c].fillna('__missing__').astype(str).values
            for c in cols_present
        }
        for c1, c2 in self.cat_pairs:
            if c1 in df.columns and c2 in df.columns:
                v1 = df[c1].fillna('__missing__').astype(str).values
                v2 = df[c2].fillna('__missing__').astype(str).values
                levels_dict[f'{c1}_{c2}'] = (v1 + '_' + v2)
        return pd.DataFrame(levels_dict, index=df.index)

    def fit_transform(self, X: pd.DataFrame, y: pd.Series) -> pd.DataFrame:
        X_out = X.copy()
        self.global_mean = float(y.mean())
        self.te_mapping = {}
        self.freq_mapping = {}

        levels_df = self._make_discrete_levels(X)
        cols_present = levels_df.columns.tolist()
        y_arr = y.to_numpy(dtype=np.float64)

        skf = StratifiedKFold(n_splits=self.n_splits, shuffle=True, random_state=self.random_state)
        splits = list(skf.split(X, y))

        for col in cols_present:
            codes, uniques = pd.factorize(levels_df[col].values)
            n_uniques = len(uniques)

            # Global mapping
            counts = np.bincount(codes, minlength=n_uniques)
            sums = np.bincount(codes, weights=y_arr, minlength=n_uniques)
            smoothed_global = (sums + self.smooth * self.global_mean) / (counts + self.smooth)
            freqs_global = counts / len(codes)

            self.te_mapping[col] = dict(zip(uniques, smoothed_global))
            self.freq_mapping[col] = dict(zip(uniques, freqs_global))

            # Fast Out-of-Fold computation
            oof_te = np.zeros(len(X), dtype=np.float32)
            oof_freq = np.zeros(len(X), dtype=np.float32)

            for tr_idx, val_idx in splits:
                tr_codes = codes[tr_idx]
                tr_y = y_arr[tr_idx]
                tr_mean = float(tr_y.mean())

                tr_counts = np.bincount(tr_codes, minlength=n_uniques)
                tr_sums = np.bincount(tr_codes, weights=tr_y, minlength=n_uniques)

                mask_observed = tr_counts > 0
                tr_smoothed = np.full(n_uniques, tr_mean, dtype=np.float32)
                tr_smoothed[mask_observed] = (tr_sums[mask_observed] + self.smooth * tr_mean) / (tr_counts[mask_observed] + self.smooth)
                tr_freqs = (tr_counts / len(tr_codes)).astype(np.float32)

                val_codes = codes[val_idx]
                oof_te[val_idx] = tr_smoothed[val_codes]
                oof_freq[val_idx] = tr_freqs[val_codes]

            X_out[f'{col}_te'] = oof_te
            X_out[f'{col}_freq'] = oof_freq

        return X_out

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        X_out = X.copy()
        levels_df = self._make_discrete_levels(X)
        for col in levels_df.columns:
            if col in self.te_mapping:
                lvl_col = levels_df[col]
                X_out[f'{col}_te'] = lvl_col.map(self.te_mapping[col]).fillna(self.global_mean).values.astype(np.float32)
                X_out[f'{col}_freq'] = lvl_col.map(self.freq_mapping[col]).fillna(0.0).values.astype(np.float32)
        return X_out


# Backward compatibility aliases
UniversalLevelTargetEncoder = DiscreteCategoricalTargetEncoder
ValueLevelTargetEncoder = DiscreteCategoricalTargetEncoder


class CompetitionSolver:
    def __init__(self, n_splits: int = 10, random_state: int = 42, use_neural_net: bool = True, n_estimators: Optional[int] = None):
        self.n_splits = n_splits
        self.random_state = random_state
        self.use_neural_net = use_neural_net
        self.n_estimators = n_estimators
        self.fold_models = []
        self.fold_encoders = []

    def cross_validate(self, X: pd.DataFrame, y: pd.Series) -> Tuple[np.ndarray, float]:
        """
        Executes a leak-free 10-fold Stratified CV loop with clean feature engineering,
        discrete target encoding, and diverse model ensembling.
        """
        X = X.reset_index(drop=True)
        y = y.reset_index(drop=True)

        cv = StratifiedKFold(n_splits=self.n_splits, shuffle=True, random_state=self.random_state)

        num_models = 4 if self.use_neural_net else 3
        oof_preds_matrix = np.zeros((len(X), num_models))
        fold_scores = []

        self.fold_models = []
        self.fold_encoders = []

        for fold, (train_idx, val_idx) in enumerate(cv.split(X, y)):
            X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
            X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]

            # 1. Clean Feature Engineering
            X_train_clean = preprocess_and_engineer(X_train)
            X_val_clean = preprocess_and_engineer(X_val)

            # 2. Leak-Free Discrete Target & Frequency Encoding
            te = DiscreteCategoricalTargetEncoder(smooth=20.0, n_splits=5, random_state=self.random_state + fold)
            X_train_clean = te.fit_transform(X_train_clean, y_train)
            X_val_clean = te.transform(X_val_clean)

            # 3. Categorical Label Encoding localized to the fold
            cat_cols = list(X_train_clean.select_dtypes(exclude=[np.number]).columns)
            encoders = {}
            for col in cat_cols:
                le = LabelEncoder()
                train_series = X_train_clean[col].fillna('Missing').astype(str)
                val_series = X_val_clean[col].fillna('Missing').astype(str)


                X_train_clean[col] = le.fit_transform(train_series).astype(np.int8)
                val_classes = list(set(val_series.tolist()))
                missing_classes = set(val_classes) - set(le.classes_)
                if missing_classes:
                    le.classes_ = np.append(le.classes_, list(missing_classes))
                X_val_clean[col] = le.transform(val_series).astype(np.int8)
                encoders[col] = le

            self.fold_encoders.append({
                'encoders': encoders,
                'target_encoder': te,
                'cat_cols': cat_cols
            })

            # 4. GBDT Training
            lgb_params, xgb_params, cat_params = get_calibrated_model_params()
            if self.n_estimators is not None:
                lgb_params["n_estimators"] = self.n_estimators
                xgb_params["n_estimators"] = self.n_estimators
                cat_params["iterations"] = self.n_estimators

            lgb = LGBMClassifier(**lgb_params)
            xgb = XGBClassifier(**xgb_params)
            cat = CatBoostClassifier(**cat_params)

            lgb.fit(X_train_clean, y_train)
            xgb.fit(X_train_clean, y_train)
            cat.fit(X_train_clean, y_train)

            p_lgb = lgb.predict_proba(X_val_clean)[:, 1]
            p_xgb = xgb.predict_proba(X_val_clean)[:, 1]
            p_cat = cat.predict_proba(X_val_clean)[:, 1]

            fold_model_dict = {'lgb': lgb, 'xgb': xgb, 'cat': cat}

            if self.use_neural_net:
                nn_epochs = 1 if (self.n_estimators is not None and self.n_estimators < 10) else 6
                nn = DeepTabularClassifier(hidden_dim=128, num_blocks=2, epochs=nn_epochs, batch_size=4096)
                nn.fit(X_train_clean, y_train.values, cat_cols=cat_cols)
                p_nn = nn.predict_proba(X_val_clean)[:, 1]
                fold_model_dict['nn'] = nn
                blend_preds = (p_lgb + p_xgb + p_cat + p_nn) / 4.0
                oof_preds_matrix[val_idx, 3] = p_nn
            else:
                blend_preds = (p_lgb + p_xgb + p_cat) / 3.0

            self.fold_models.append(fold_model_dict)

            oof_preds_matrix[val_idx, 0] = p_lgb
            oof_preds_matrix[val_idx, 1] = p_xgb
            oof_preds_matrix[val_idx, 2] = p_cat

            fold_auc = roc_auc_score(y_val, blend_preds)
            fold_scores.append(fold_auc)

        mean_auc = float(np.mean(fold_scores))
        return oof_preds_matrix, mean_auc


def to_percentile_rank(predictions: np.ndarray) -> np.ndarray:
    """
    Converts raw probability predictions to percentile ranks [0, 1].
    Preserves exact ordinality, eliminates distribution discrepancy across models without introducing ties.
    """
    return (rankdata(predictions) - 0.5) / len(predictions)


def to_gauss_rank(ranks: np.ndarray, eps: float = 1e-6) -> np.ndarray:
    """Transforms rank percentiles (0, 1) into standard Gaussian domain using probit."""
    clipped = np.clip(ranks, eps, 1.0 - eps)
    return norm.ppf(clipped)


def perform_ks_drift_screen(oof_rank: np.ndarray, test_rank: np.ndarray, threshold: float = 0.05) -> Tuple[bool, float]:
    """Kolmogorov-Smirnov two-sample test to detect rank distribution drift between OOF and Test sets."""
    stat, p_val = ks_2samp(oof_rank, test_rank)
    passed = bool(stat <= threshold)
    return passed, float(stat)


class LogisticStacker:
    """Logistic Regression Stacker on Gauss-Rank Transformed Percentiles."""
    def __init__(self, C: float = 0.03, random_state: int = 42):
        self.C = C
        self.random_state = random_state
        self.model = LogisticRegression(C=self.C, max_iter=1000, random_state=self.random_state)
        self.coef_ = None
        self.intercept_ = None

    def fit(self, preds_matrix: np.ndarray, y: np.ndarray):
        self.model.fit(preds_matrix, y)
        self.coef_ = self.model.coef_[0]
        self.intercept_ = float(self.model.intercept_[0])
        return self

    def predict_proba(self, preds_matrix: np.ndarray) -> np.ndarray:
        return self.model.predict_proba(preds_matrix)[:, 1]


class NelderMeadRankStacker:
    """Non-Parametric Nelder-Mead Rank-AUC Optimizer with Softmax Projection."""
    def __init__(self, random_state: int = 42):
        self.random_state = random_state
        self.weights_ = None

    def _objective(self, unconstrained_weights: np.ndarray, preds_matrix: np.ndarray, y: np.ndarray) -> float:
        exp_w = np.exp(unconstrained_weights - np.max(unconstrained_weights))
        weights = exp_w / np.sum(exp_w)
        blended = np.dot(preds_matrix, weights)
        return -roc_auc_score(y, blended)

    def fit(self, preds_matrix: np.ndarray, y: np.ndarray):
        n_models = preds_matrix.shape[1]
        init_weights = np.zeros(n_models)

        res = minimize(
            self._objective,
            x0=init_weights,
            args=(preds_matrix, y),
            method='Nelder-Mead',
            options={'maxiter': 500, 'xatol': 1e-4, 'fatol': 1e-5}
        )

        exp_w = np.exp(res.x - np.max(res.x))
        self.weights_ = exp_w / np.sum(exp_w)
        return self

    def predict_proba(self, preds_matrix: np.ndarray) -> np.ndarray:
        if self.weights_ is None:
            raise ValueError("NelderMeadRankStacker is not fitted yet.")
        return np.dot(preds_matrix, self.weights_)


class EnsembleBlender:
    """Two-Stage Robust Blending Stacker."""
    def __init__(self, random_state: int = 42):
        self.random_state = random_state
        self.tree_weights_ = None
        self.alpha_ = None

    def _to_rank(self, p: np.ndarray) -> np.ndarray:
        return (rankdata(p) - 0.5) / len(p)

    def fit(self, oof_lgb: np.ndarray, oof_cat: np.ndarray, oof_xgb: np.ndarray, oof_nn: np.ndarray, y: np.ndarray):
        r_lgb = self._to_rank(oof_lgb)
        r_cat = self._to_rank(oof_cat)
        r_xgb = self._to_rank(oof_xgb)
        r_nn = self._to_rank(oof_nn)
        def objective(params):
            w1, w2, w3, alpha = params
            w_sum = w1 + w2 + w3 + 1e-8
            w1_n, w2_n, w3_n = w1 / w_sum, w2 / w_sum, w3 / w_sum

            r_tree = w1_n * r_lgb + w2_n * r_cat + w3_n * r_xgb
            p_final = alpha * r_tree + (1.0 - alpha) * r_nn

            # Tie-breaking surrogate loss: Loss = -AUC + 10^-4 * LogLoss
            from sklearn.metrics import log_loss
            # Log loss requires valid probability distributions, so we clip p_final
            p_clipped = np.clip(p_final, 1e-7, 1.0 - 1e-7)
            auc_score = roc_auc_score(y, p_clipped)
            logloss = log_loss(y, p_clipped)
            return -auc_score + 1e-4 * logloss


        init_params = [0.33, 0.33, 0.33, 0.8]
        bounds = [(0, 1), (0, 1), (0, 1), (0, 1)]
        res = minimize(objective, init_params, method='Nelder-Mead', bounds=bounds)

        w1, w2, w3, alpha = res.x
        w_sum = w1 + w2 + w3 + 1e-8
        self.tree_weights_ = np.array([w1 / w_sum, w2 / w_sum, w3 / w_sum], dtype=np.float32)
        self.alpha_ = float(alpha)
        return self

    def predict_proba(self, p_lgb: np.ndarray, p_cat: np.ndarray, p_xgb: np.ndarray, p_nn: np.ndarray) -> np.ndarray:
        if self.tree_weights_ is None:
            raise ValueError("TwoStageHybridStacker is not fitted yet.")
        r_lgb = self._to_rank(p_lgb)
        r_cat = self._to_rank(p_cat)
        r_xgb = self._to_rank(p_xgb)
        r_nn = self._to_rank(p_nn)

        r_tree = self.tree_weights_[0] * r_lgb + self.tree_weights_[1] * r_cat + self.tree_weights_[2] * r_xgb
        p_final = self.alpha_ * r_tree + (1.0 - self.alpha_) * r_nn
        return p_final


### Training Loop (10-Fold Stratified CV)


In [ ]:
import os
import sys


import os
import numpy as np
import pandas as pd
import scipy.stats
import joblib
from sklearn.metrics import roc_auc_score

def resolve_data_path(filename):
    import zipfile
    paths_to_check = [
        f"/content/{filename}",
        f"/content/data/{filename}",
        f"/content/playground-series-s6e8/{filename}",
        f"/kaggle/input/playground-series-s6e8/{filename}",
        f"/kaggle/input/competitions/playground-series-s6e8/{filename}",
        f"../input/playground-series-s6e8/{filename}",
        f"data/{filename}",
        f"./{filename}",
        f"../{filename}"
    ]
    for path in paths_to_check:
        if os.path.exists(path):
            print(f"[INFO] Successfully resolved {filename} to: {path}", flush=True)
            return path

    # Auto-extract zip if found in /content, data, or current dir
    for zip_candidate in ["playground-series-s6e8.zip", "data/playground-series-s6e8.zip", "/content/playground-series-s6e8.zip", "/content/data/playground-series-s6e8.zip"]:
        if os.path.exists(zip_candidate):
            print(f"[INFO] Found zip archive {zip_candidate}, auto-extracting to data/...", flush=True)
            os.makedirs("data", exist_ok=True)
            with zipfile.ZipFile(zip_candidate, 'r') as zip_ref:
                zip_ref.extractall("data")
            if os.path.exists(f"data/{filename}"):
                return f"data/{filename}"
            if os.path.exists(filename):
                return filename

    # Global recursive search in /content, /kaggle/input, data, .
    search_roots = ["/content", "/kaggle/input", "../input", "data", "."]
    for root_dir in search_roots:
        if os.path.exists(root_dir):
            for root, dirs, files in os.walk(root_dir):
                if filename in files:
                    found = os.path.join(root, filename)
                    print(f"[INFO] Found {filename} via walk: {found}", flush=True)
                    return found

    # If Kaggle API is configured in Colab, attempt automated direct download
    try:
        import subprocess
        print(f"[INFO] Attempting automated Kaggle download for {filename}...", flush=True)
        os.makedirs("data", exist_ok=True)
        subprocess.run(["kaggle", "competitions", "download", "-c", "playground-series-s6e8", "-p", "data/"], check=False)
        for z in os.listdir("data"):
            if z.endswith(".zip"):
                with zipfile.ZipFile(os.path.join("data", z), 'r') as zip_ref:
                    zip_ref.extractall("data")
        if os.path.exists(f"data/{filename}"):
            return f"data/{filename}"
    except Exception as e:
        print(f"[WARN] Auto-download failed: {e}", flush=True)

    raise FileNotFoundError(f"Could not find {filename} anywhere in {search_roots}")

def main():
    print("Loading training data...", flush=True)
    train_path = resolve_data_path("train.csv")

    df_train = pd.read_csv(train_path)

    target_col = "addicted_label"
    if target_col not in df_train.columns:
        raise ValueError(f"Target column '{target_col}' not found in training data")

    X = df_train.drop(columns=["id", target_col], errors="ignore")
    y = df_train[target_col]

    print(f"Training shapes -> X: {X.shape}, y: {y.shape}", flush=True)

    # Initialize 10-fold CV with 4-way modeling
    solver = CompetitionSolver(n_splits=10, random_state=42, use_neural_net=True)

    print("Starting 10-fold Stratified Cross-Validation (LGB + XGB + CAT + PyTorch NN)...", flush=True)
    oof_preds_matrix, mean_auc = solver.cross_validate(X, y)

    print(f"==================================================", flush=True)
    print(f"Baseline (Average 4-Way) OOF ROC AUC Score: {mean_auc:.5f}", flush=True)
    print(f"==================================================", flush=True)

    print("Converting OOF predictions to Gauss-Rank normal percentiles...", flush=True)
    rank_oof = np.zeros_like(oof_preds_matrix)
    for i in range(oof_preds_matrix.shape[1]):
        preds = oof_preds_matrix[:, i]
        percentiles = (scipy.stats.rankdata(preds) - 0.5) / len(preds)
        rank_oof[:, i] = to_gauss_rank(percentiles)

    print("Fitting Nested Logistic Stacker on 4-Way Gauss-Rank Percentiles...", flush=True)
    stacker = LogisticStacker(C=0.03, random_state=42)
    stacker.fit(rank_oof, y.values)

    stacked_oof_preds = stacker.predict_proba(rank_oof)
    stacked_auc = roc_auc_score(y.values, stacked_oof_preds)

    print(f"==================================================", flush=True)
    print(f"🚀 Version 6 Logistic Stack OOF ROC AUC Score: {stacked_auc:.5f}", flush=True)
    print(f"Stacker Coefficients [LGB, XGB, CAT, NN]: {stacker.coef_}", flush=True)
    print(f"Stacker Intercept: {stacker.intercept_:.5f}", flush=True)
    print(f"==================================================", flush=True)

    # Save artifact
    artifact = {
        'fold_models': solver.fold_models,
        'fold_encoders': solver.fold_encoders,
        'stacker': stacker,
        'oof_ranks': rank_oof
    }

    models_dir = "models"
    os.makedirs(models_dir, exist_ok=True)
    artifact_path = os.path.join(models_dir, "ensemble_pipeline.joblib")

    print(f"Saving ensemble pipeline artifact to {artifact_path}...", flush=True)
    joblib.dump(artifact, artifact_path)
    print("Done!", flush=True)

if __name__ == "__main__":
    main()

if __name__ == '__main__':
    main()


### Inference, KS Drift Screening and Submission Formatting


In [ ]:
import os
import sys


import os
import numpy as np
import pandas as pd
import scipy.stats
import joblib

def resolve_data_path(filename):
    paths_to_check = [
        f"/kaggle/input/playground-series-s6e8/{filename}",
        f"/kaggle/input/competitions/playground-series-s6e8/{filename}",
        f"../input/playground-series-s6e8/{filename}",
        f"data/{filename}",
        f"./{filename}",
        f"../{filename}"
    ]
    for path in paths_to_check:
        if os.path.exists(path):
            print(f"[INFO] Successfully resolved {filename} to: {path}")
            return path

    search_roots = ["/kaggle/input", "../input", "data", "."]
    for root_dir in search_roots:
        if os.path.exists(root_dir):
            for root, dirs, files in os.walk(root_dir):
                if filename in files:
                    found = os.path.join(root, filename)
                    print(f"[INFO] Found {filename} via walk: {found}")
                    return found

    raise FileNotFoundError(f"Could not find {filename} anywhere in {search_roots}")

def main():
    print("Loading test data...")
    test_path = resolve_data_path("test.csv")

    df_test = pd.read_csv(test_path)

    test_ids = df_test["id"].copy()
    X_test = df_test.drop(columns=["id"], errors="ignore")

    print(f"Test shape: {X_test.shape}")

    models_dir = "models"
    artifact_path = os.path.join(models_dir, "ensemble_pipeline.joblib")
    if not os.path.exists(artifact_path):
        raise FileNotFoundError(f"Pipeline artifact not found at {artifact_path}. Did you run train.py?")

    print("Loading ensemble pipeline artifact...")
    artifact = joblib.load(artifact_path)

    fold_models = artifact['fold_models']
    fold_encoders = artifact['fold_encoders']
    stacker = artifact.get('stacker')
    oof_ranks = artifact.get('oof_ranks')

    num_folds = len(fold_models)
    has_nn = 'nn' in fold_models[0]
    num_models = 4 if has_nn else 3
    print(f"Loaded {num_folds} folds ({num_models}-way modeling) from artifact.")

    p_lgb_folds = np.zeros((len(X_test), num_folds))
    p_xgb_folds = np.zeros((len(X_test), num_folds))
    p_cat_folds = np.zeros((len(X_test), num_folds))
    if has_nn:
        p_nn_folds = np.zeros((len(X_test), num_folds))

    print("Engineering base features for test dataset...")
    X_test_clean_base = preprocess_and_engineer(X_test)

    for fold in range(num_folds):
        print(f"Processing Fold {fold + 1}/{num_folds}...")
        X_test_clean = X_test_clean_base.copy()

        fold_artifacts = fold_encoders[fold]
        encoders = fold_artifacts['encoders']
        target_encoder = fold_artifacts.get('target_encoder')

        if target_encoder is not None:
            X_test_clean = target_encoder.transform(X_test_clean)

        for col, le in encoders.items():
            if col in X_test_clean.columns:
                test_series = X_test_clean[col].fillna('Missing').astype(str)
                test_classes = list(set(test_series.tolist()))
                missing_classes = set(test_classes) - set(le.classes_)
                if missing_classes:
                    le.classes_ = np.append(le.classes_, list(missing_classes))
                X_test_clean[col] = le.transform(test_series)

        models = fold_models[fold]
        p_lgb_folds[:, fold] = models['lgb'].predict_proba(X_test_clean)[:, 1]
        p_xgb_folds[:, fold] = models['xgb'].predict_proba(X_test_clean)[:, 1]
        p_cat_folds[:, fold] = models['cat'].predict_proba(X_test_clean)[:, 1]
        if has_nn:
            p_nn_folds[:, fold] = models['nn'].predict_proba(X_test_clean)[:, 1]

    p_lgb_mean = np.mean(p_lgb_folds, axis=1)
    p_xgb_mean = np.mean(p_xgb_folds, axis=1)
    p_cat_mean = np.mean(p_cat_folds, axis=1)

    if has_nn:
        p_nn_mean = np.mean(p_nn_folds, axis=1)
        test_preds_matrix = np.column_stack((p_lgb_mean, p_xgb_mean, p_cat_mean, p_nn_mean))
    else:
        test_preds_matrix = np.column_stack((p_lgb_mean, p_xgb_mean, p_cat_mean))

    print("Converting test predictions to Gauss-Rank normal percentiles...")
    rank_test = np.zeros_like(test_preds_matrix)
    for i in range(test_preds_matrix.shape[1]):
        preds = test_preds_matrix[:, i]
        percentiles = (scipy.stats.rankdata(preds) - 0.5) / len(preds)
        rank_test[:, i] = to_gauss_rank(percentiles)

    # Kolmogorov-Smirnov Drift Screening
    if oof_ranks is not None:
        model_names = ['LightGBM', 'XGBoost', 'CatBoost', 'PyTorch NN'][:test_preds_matrix.shape[1]]
        for i, name in enumerate(model_names):
            passed, stat = perform_ks_drift_screen(oof_ranks[:, i], rank_test[:, i])
            status_str = "PASSED ✅" if passed else "WARNING ⚠️"
            print(f"[KS-Drift Screen] {name}: stat={stat:.4f} -> {status_str}")

    if stacker is not None:
        print("Applying Nested Logistic Stacker...")
        final_preds = stacker.predict_proba(rank_test)
    else:
        final_preds = np.mean(rank_test, axis=1)

    print("Generating submission file...")
    submission = pd.DataFrame({
        "id": test_ids,
        "addicted_label": final_preds
    })

    assert submission.shape[0] == df_test.shape[0], "Shape mismatch: submission rows != test rows"
    assert submission.shape[1] == 2, "Submission must have exactly 2 columns"
    assert not submission.isnull().values.any(), "Submission contains NaN values"
    assert submission['addicted_label'].min() >= 0.0, "Probabilities < 0.0 found"
    assert submission['addicted_label'].max() <= 1.0, "Probabilities > 1.0 found"

    outputs_dir = "outputs"
    os.makedirs(outputs_dir, exist_ok=True)
    sub_path = os.path.join(outputs_dir, "submission.csv")

    submission.to_csv(sub_path, index=False)
    submission.to_csv("submission.csv", index=False)
    print(f"Sanity checks passed. Final submission saved to {sub_path} and submission.csv")

if __name__ == "__main__":
    main()

if __name__ == '__main__':
    main()


### Direct Cloud-to-Competition Submission (Zero Local Roundtrips)


In [ ]:
# Auto-submit directly from cloud environment to Kaggle
import subprocess
sub_file = "submission.csv" if os.path.exists("submission.csv") else "outputs/submission.csv"
if os.path.exists(sub_file):
    print(f"🚀 Submitting {sub_file} directly from Cloud to Kaggle Competition...", flush=True)
    res = subprocess.run([
        "kaggle", "competitions", "submit",
        "-c", "playground-series-s6e8",
        "-f", sub_file,
        "-m", "Elite 10-Fold 4-Way GPU Ensemble (LGB+XGB+CAT+NN + Stacker)"
    ], capture_output=True, text=True)
    print(res.stdout, flush=True)
    if res.stderr:
        print("Kaggle CLI response:", res.stderr, flush=True)
else:
    print(f"Submission file not found at {sub_file}")
